In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

# ------------------------- 1. ??c v? x? l? d? li?u -------------------------
df = pd.read_csv(r'D:\dow\project\data\full_dataset_with_gan.csv')

def add_features(df):
    df = df.copy()
    base = [
        'Normalized_Temp', 'Normalized_Vibration', 'Normalized_Pressure',
        'Normalized_Voltage', 'Normalized_Current',
        'FFT_Feature1', 'FFT_Feature2', 'Anomaly_Score'
    ]
    df = df.sort_values(['Sensor_ID', 'Year', 'Month', 'Day', 'Hour', 'Minute'])
    for col in base:
        for w in [3, 7, 15]:
            df[f'{col}_rollmean{w}'] = df.groupby('Sensor_ID')[col].transform(lambda x: x.rolling(w, min_periods=1).mean())
            df[f'{col}_rollstd{w}'] = df.groupby('Sensor_ID')[col].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))
    df['Temp_Vibration_ratio'] = df['Normalized_Temp'] / (df['Normalized_Vibration'] + 1e-5)
    df['Delta_Current'] = df['Normalized_Current'].diff().fillna(0)
    return df

df = add_features(df)
drop_cols = ['Sensor_ID', 'Fault_Status', 'Fault_Type', 'Year','Month','Day','Hour','Minute']
feature_cols = [c for c in df.columns if c not in drop_cols]

# ------------------------- 2. H?m ??nh gi? -------------------------
def print_metrics(y_true, y_pred, name):
    print(f"\n=== [{name}] ===")
    print("Classification Report:")
    print(classification_report(y_true, y_pred, digits=4))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Macro F1:", f1_score(y_true, y_pred, average='macro'))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))

# ------------------------- 3. H?m ch?y t?ng k?ch b?n -------------------------
def run_case(case_name, X_train, y_train, X_test, y_test):
    print(f"\n===== {case_name} =====")
    
    # XGBoost
    xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    xgb.fit(X_train, y_train)
    y_pred_xgb = xgb.predict(X_test)
    print_metrics(y_test, y_pred_xgb, "XGBoost")

    # LightGBM
    lgbm = LGBMClassifier(random_state=42)
    lgbm.fit(X_train, y_train)
    y_pred_lgbm = lgbm.predict(X_test)
    print_metrics(y_test, y_pred_lgbm, "LightGBM")

# ------------------------- 4. H?m chia theo thi?t b? (ph? thu?c) -------------------------
def split_by_device(df, test_size=0.15, val_size=0.15):
    train_list, test_list, val_list = [], [], []
    for device in df['Sensor_ID'].unique():
        sub = df[df['Sensor_ID'] == device]
        train_sub, test_val_sub = train_test_split(sub, test_size=(test_size + val_size), random_state=42, shuffle=True)
        test_sub, val_sub = train_test_split(test_val_sub, test_size=val_size / (test_size + val_size), random_state=42)
        train_list.append(train_sub)
        test_list.append(test_sub)
        val_list.append(val_sub)
    train = pd.concat(train_list)
    test = pd.concat(test_list)
    val = pd.concat(val_list)
    return train, test, val



In [ ]:
# ====================== 5. C?c k?ch b?n chia d? li?u ======================

## CASE 1: Dependent split per device
train, test, val = split_by_device(df)
X_train, y_train = train[feature_cols], train['Fault_Status']
X_test, y_test = test[feature_cols], test['Fault_Status']
run_case("CASE 1: Dependent split per device", X_train, y_train, X_test, y_test)


In [ ]:
## CASE 2: Independent split per device
unique_sensors = df['Sensor_ID'].unique()
np.random.seed(42)
np.random.shuffle(unique_sensors)
n = len(unique_sensors)
train_sensor = unique_sensors[:int(0.7*n)]
test_sensor = unique_sensors[int(0.7*n):int(0.85*n)]
train = df[df['Sensor_ID'].isin(train_sensor)]
test = df[df['Sensor_ID'].isin(test_sensor)]
X_train, y_train = train[feature_cols], train['Fault_Status']
X_test, y_test = test[feature_cols], test['Fault_Status']
run_case("CASE 2: Independent split per device", X_train, y_train, X_test, y_test)


In [ ]:
## CASE 3: Random split all data
X = df[feature_cols]
y = df['Fault_Status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, shuffle=True)
run_case("CASE 3: Random split all data", X_train, y_train, X_test, y_test)
